> **Current notebook role — 2026-11-12.** This notebook retains the upstream stage code used by the manuscript implementation. Selected figure production uses `run.py` in `code_release/iCCA-molecular-subtyping` or the public source repository. See `docs/NOTEBOOK_STATUS.md` in that code repository for stage ownership. Earlier execution outputs are preserved in the archived notebook snapshot.

# Step 3 — Observed large-/small-duct features and classification

## tl;dr

Pathology-defined duct labels are available for 115 tumors from three studies (29 large-duct and 86 small-duct). The rebuilt model uses outer-fold feature screening, inner-fold LASSO tuning, repeated out-of-fold prediction, and leave-one-study-out evaluation. The conservative locked-corrected-expression analysis reaches AUROC 0.803 and AUPRC 0.647 in repeated nested CV; pooled leave-one-study-out AUROC is 0.744. These estimates replace the historical training-set resubstitution metrics.

Among 93 principal-state tumors with observed pathology labels, large-duct morphology is enriched in C3 relative to C1/C2 (odds ratio 8.49, 95% CI 2.75–29.15). Observed pathology, cross-fitted predictions, and full-model projections remain separate fields and denominators.

## Context & Methods

This notebook characterizes observed large-/small-duct expression features and evaluates a duct-type LASSO model. It is not the C1/C2/C3 subtype classifier, which remains owned by Step 4.

### Key assumptions and guardrails

- Step 1 expression matrices and subtype assignments are locked read-only inputs.
- Positive differential-expression effects mean **large-duct minus small-duct** and include Study as a covariate.
- Variance screening, scaling, class weighting, and lambda selection are fitted using training data only.
- Headline performance uses repeated nested-CV out-of-fold predictions and leave-one-study-out predictions, never training-set resubstitution.
- The 2019Ahn source used bile duct-type/cholangiolar-type terminology; its project-level mapping to large-/small-duct requires an explicit harmonization caveat.
- Survival analysis is deferred to the later clinical gate.

In [ ]:
rm(list = ls())
options(stringsAsFactors = FALSE)
project_root <- normalizePath(getwd())
step_dir <- file.path(project_root, "3-DuctType")
table_dir <- file.path(step_dir, "publication", "tables")
figure_dir <- file.path(step_dir, "publication", "figures")
refresh_step3 <- identical(Sys.getenv("SCLUCID_STEP3_REFRESH", "0"), "1")
required <- c("IRdisplay", "readxl")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) stop("Missing R packages: ", paste(missing, collapse = ", "))
cat("Execution mode:", ifelse(refresh_step3, "full Step3 refresh", "validation/read-only"), "
")

In [ ]:
if (refresh_step3) {
  status <- system2("Rscript", file.path("3-DuctType", "scripts", "run_step3_duct_analysis.R"))
  if (status != 0) stop("Step3 analysis script failed with status ", status)
}
required_outputs <- file.path(table_dir, c(
  "step3_input_contract.csv", "step3_pathology_label_provenance.csv",
  "step3_study_adjusted_deg_all_genes.csv", "step3_classifier_performance_summary.csv",
  "step3_nested_cv_oof_predictions.csv", "step3_leave_one_study_out_metrics.csv",
  "step3_all_samples_observed_and_predicted_duct.csv", "step3_release_gate_status.csv"
))
stopifnot(all(file.exists(required_outputs)), all(file.info(required_outputs)$size > 0))

## Data

### 1. Locked inputs and sample eligibility

In [ ]:
input_contract <- read.csv(file.path(table_dir, "step3_input_contract.csv"), check.names = FALSE)
label_counts <- read.csv(file.path(table_dir, "step3_label_provenance_by_study.csv"), check.names = FALSE)
eligibility <- read.csv(file.path(table_dir, "step3_sample_eligibility.csv"), check.names = FALSE)
stopifnot(sum(eligibility$EligibleForObservedDuctAnalysis) == 115)
input_contract
label_counts

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_label_provenance.png'))

### 2. Pathology-label provenance

The three cohorts all contain both classes, but terminology is not perfectly homogeneous. Deng et al. used large versus small intrahepatic bile-duct subtypes based on the affected duct level/size, and Lee et al. used the WHO fifth-edition histological framework. Ahn et al. reported bile duct-type versus cholangiolar-type pathology; the project metadata maps these labels to large-/small-duct, but sample-level pathological equivalence was not independently re-reviewed.

In [ ]:
pathology_provenance <- read.csv(file.path(table_dir, "step3_pathology_label_provenance.csv"), check.names = FALSE)
pathology_provenance

## Results

### 3. Study-adjusted duct-type expression features

In [ ]:
deg <- read.csv(file.path(table_dir, "step3_study_adjusted_deg_all_genes.csv"), check.names = FALSE)
meta_deg <- read.csv(file.path(table_dir, "step3_cross_study_deg_meta_summary.csv"), check.names = FALSE)
data.frame(
  Direction = c("Large-duct higher", "Small-duct higher"),
  Genes = c(sum(deg$FDR < 0.05 & deg$logFC_large_minus_small >= 0.5),
            sum(deg$FDR < 0.05 & deg$logFC_large_minus_small <= -0.5))
)
head(deg, 15)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_study_adjusted_deg_volcano.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_study_adjusted_marker_heatmap.png'))

### 4. Ranked pathway characterization

In [ ]:
gsea <- read.csv(file.path(table_dir, "step3_ranked_gsea_large_minus_small.csv"), check.names = FALSE)
gsea_sig <- gsea[!is.na(gsea$padj) & gsea$padj < 0.05, ]
head(gsea_sig[, c("pathway", "Collection", "Direction", "NES", "padj", "size")], 15)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_ranked_gsea.png'))

### 5. Leakage-control contract

In [ ]:
modeling_contract <- read.csv(file.path(table_dir, "step3_modeling_contract.csv"), check.names = FALSE)
legacy_audit <- read.csv(file.path(table_dir, "step3_legacy_static_audit_findings.csv"), check.names = FALSE)
modeling_contract
legacy_audit

### 6. Repeated nested-CV discrimination and calibration

In [ ]:
performance <- read.csv(file.path(table_dir, "step3_classifier_performance_summary.csv"), check.names = FALSE)
primary <- performance[performance$Analysis == "Repeated nested CV | locked corrected expression", ]
primary

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_nested_cv_discrimination.png'))

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_nested_cv_calibration.png'))

The uncorrected-expression sensitivity produces higher apparent performance than the locked corrected representation. Because uncorrected expression can retain study/platform signal, it is disclosed as a confounding sensitivity result and is not promoted as the headline model.

### 7. Leave-one-study-out transfer

In [ ]:
loso_metrics <- read.csv(file.path(table_dir, "step3_leave_one_study_out_metrics.csv"), check.names = FALSE)
loso_metrics[loso_metrics$Analysis == "LOSO | locked corrected expression" &
               loso_metrics$Metric %in% c("AUROC", "AUPRC", "BalancedAccuracy", "Sensitivity", "Specificity"), ]

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_leave_one_study_out_performance.png'))

### 8. Feature stability and final projection interface

In [ ]:
feature_stability <- read.csv(file.path(table_dir, "step3_nested_cv_feature_stability.csv"), check.names = FALSE)
final_coefficients <- read.csv(file.path(table_dir, "step3_final_lasso_nonzero_coefficients.csv"), check.names = FALSE)
head(feature_stability, 20)
final_coefficients

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_nested_cv_feature_stability.png'))

In [ ]:
duct_interface <- read.csv(file.path(table_dir, "step3_all_samples_observed_and_predicted_duct.csv"), check.names = FALSE)
stopifnot(all(c("DuctObserved", "DuctOOFProbabilityLarge", "DuctProjectedProbabilityLargeFullModel",
                "DuctPredictionSource") %in% names(duct_interface)))
table(duct_interface$DuctPredictionSource)
table(duct_interface$DuctObserved, useNA = "ifany")

### 9. Observed pathology association with principal molecular states

In [ ]:
observed_distribution <- read.csv(file.path(table_dir, "step3_observed_duct_distribution_by_subtype.csv"), check.names = FALSE)
observed_tests <- read.csv(file.path(table_dir, "step3_observed_duct_subtype_association_tests.csv"), check.names = FALSE)
observed_distribution[observed_distribution$FinalSubtype %in% c("C1", "C2", "C3"), ]
observed_tests

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, 'step3_observed_duct_by_main_subtype.png'))

### 10. Supportive clinical audit

Clinical associations below use observed pathology labels only. They are descriptive/supportive because availability and definitions vary by study. Predicted duct type is not substituted for missing pathology. Survival analysis is not performed here.

In [ ]:
clinical_availability <- read.csv(file.path(table_dir, "step3_observed_clinical_availability.csv"), check.names = FALSE)
clinical_tests <- read.csv(file.path(table_dir, "step3_observed_clinical_association_tests.csv"), check.names = FALSE)
clinical_availability
clinical_tests

## Takeaways

- Step 3 now provides leakage-controlled, patient-level duct classification evidence rather than training-set fit statistics.
- The corrected-expression model has moderate cross-study discrimination; performance is sufficient for hypothesis-generating projection but not a standalone clinical diagnostic claim.
- C3 is strongly enriched for observed large-duct morphology, while duct type and transcriptomic subtype remain non-interchangeable variables.
- Uncorrected-expression superiority and the 2019Ahn terminology mapping are mandatory caveats.
- The all-sample prediction table keeps observed pathology, repeated-CV OOF predictions, and full-model projections in separate columns.

## Release gate

In [ ]:
gate <- read.csv(file.path(table_dir, "step3_release_gate_status.csv"), check.names = FALSE)
gate
stopifnot(!any(gate$Blocking & !grepl("^PASS", gate$Status)))
manifest <- read.csv(file.path(table_dir, "step3_publication_output_manifest.csv"), check.names = FALSE)
stopifnot(all(manifest$Exists), all(manifest$NonEmpty))
cat("Step 3 release gate: 0 blocking failures;", nrow(manifest), "validated artifacts.
")
sessionInfo()